# RAG (Retrieval-Augmented Generation)
### 3º Encontro do Grupo de Estudos NLP | INF-UFG / CEIA

Este notebook acompanha a apresentação do 3º encontro.

Cada bloco de código vem acompanhado de explicações em Markdown, portanto basta ler e executar na ordem.

## 🔗 Material da Apresentação

| Recurso | Link |
|---------|------|
| Slides  | _https://www.canva.com/design/DAHEDiC9DWg/wfbuVOayU_xTO13VVc8mlA/view?_ |
| Paper — Lewis et al. (2021) | _https://arxiv.org/pdf/2005.11401_ |
| Vídeo — Computerfile: Vector Search | _https://www.youtube.com/watch?v=YDdKiQNw80c&t=500s_ |

---

## ⚙️ Configuração Inicial

Execute as **duas células abaixo uma única vez** no início da sessão.

In [ ]:
# Instalação das dependências
# ⏳ Pode levar ~1-2 minutos na primeira vez
!pip install google-generativeai sentence-transformers chromadb pymupdf scikit-learn numpy -q

print("✅ Dependências instaladas!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [ ]:
# Importações
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import chromadb
import fitz          # PyMuPDF
import google.generativeai as genai
import warnings
warnings.filterwarnings("ignore")

print("✅ Bibliotecas importadas!")

✅ Bibliotecas importadas!


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# Configuração da API do Google (Gemini)
# -------------------------------------------------
# COMO OBTER SUA CHAVE GRATUITA:
#   1. Acesse https://aistudio.google.com/
#   2. Clique em "Get API key"
#   3. Crie uma chave e copie
#
# COMO USAR NO COLAB (recomendado — sem expor a chave):
#   1. Clique no ícone 🔑 na barra lateral esquerda ("Secrets")
#   2. Adicione um secret com nome: GOOGLE_API_KEY
#   3. Cole o valor da chave
#   4. Ative o acesso ao notebook
# -------------------------------------------------

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    print("✅ Chave carregada dos Secrets do Colab")
except Exception:
    import getpass
    GOOGLE_API_KEY = getpass.getpass("Cole sua Google API Key aqui: ")
    print("✅ Chave inserida")

# Configurar o cliente Gemini
genai.configure(api_key=GOOGLE_API_KEY)
model_llm = genai.GenerativeModel("gemini-2.5-flash")

print("🤖 Modelo Gemini 2.5 Flash pronto!")

✅ Chave carregada dos Secrets do Colab
🤖 Modelo Gemini 2.5 Flash pronto!


---

## Parte 1: O Problema — LLMs e Alucinação

**Cenário:** você foi contratado para construir um assistente inteligente para uma empresa. Ele deve responder perguntas sobre processos internos — RH, contratos, normas, portarias.

Você acabou de aprender a usar a API de um LLM. Então... vamos tentar!

### 🔴 Bloco 1 — A primeira tentativa: LLM sem contexto

Chamada direta ao modelo. Sem nenhuma informação adicional.

In [ ]:
# Bloco 1 — Nosso assistente (versão 1.0)
# Chamada simples: só a pergunta, sem contexto

pergunta_v1 = "Qual é a política de férias da Empresa Semantix para funcionários CLT?"

resposta_v1 = model_llm.generate_content(pergunta_v1)

print("❓ Pergunta:", pergunta_v1)
print()
print("🤖 Resposta do modelo:")
print("─" * 60)
print(resposta_v1.text)

❓ Pergunta: Qual é a política de férias da Empresa Semantix para funcionários CLT?

🤖 Resposta do modelo:
────────────────────────────────────────────────────────────
Como uma inteligência artificial, não tenho acesso direto às políticas internas e confidenciais de Recursos Humanos da Semantix ou de qualquer outra empresa específica. Essas informações são de caráter privado da organização.

No entanto, posso explicar a política geral de férias para funcionários CLT no Brasil, que a Semantix, como qualquer empresa brasileira, *deve* seguir. A política interna da Semantix provavelmente complementa essas regras com detalhes sobre o agendamento, comunicação e outros procedimentos específicos da empresa.

**Direitos e Regras Gerais de Férias para Funcionários CLT (que a Semantix segue):**

1.  **Período Aquisitivo:**
    *   Após cada período de 12 meses de trabalho (o "período aquisitivo"), o empregado tem direito a 30 dias corridos de férias remuneradas.

2.  **Período Concessivo:**
    *

**O que aconteceu?**

O modelo respondeu de forma coerente e bem escrita — mas usando conhecimento **geral sobre a CLT brasileira**, não as políticas específicas da Empresa Semantix. Ele não tem como saber o que está nos documentos internos da empresa.

Isso inclusive pode levar à **alucinação**. O modelo não sabe que não sabe. Ele gera uma resposta plausível com base nos padrões do treinamento.

> 💭 **Por que o modelo errou?**
>
> Um LLM é treinado em bilhões de textos da internet e fica "congelado" após o treino. Ele nunca vai saber a política interna da Empresa porque isso nunca esteve na internet.

### ✅ Bloco 2 — Injetando o contexto manualmente

E se, em vez de deixar o modelo "adivinhar", a gente **fornecer o trecho relevante** como parte do prompt?

In [ ]:
# Bloco 2 — Nosso assistente (versão 2.0)
# Injetamos manualmente o contexto relevante no prompt

contexto_manual = """
POLÍTICA DE FÉRIAS — EMPRESA SEMANTIX
Conforme a Portaria Interna nº 42/2024:

- Funcionários CLT têm direito a 30 dias corridos de férias por ano.
- As férias podem ser fracionadas em até 3 períodos, sendo o menor deles
  não inferior a 14 dias.
- O período de gozo deve ser comunicado ao RH com 30 dias de antecedência.
- O pagamento ocorre até 2 dias antes do início das férias.
"""

pergunta_v2 = "Qual é a política de férias da Empresa SEMANTIX para funcionários CLT?"

prompt_com_contexto = f"""Responda a pergunta abaixo usando APENAS as informações do contexto fornecido.
Se a resposta não estiver no contexto, diga que não sabe.

CONTEXTO:
{contexto_manual}

PERGUNTA:
{pergunta_v2}"""

resposta_v2 = model_llm.generate_content(prompt_com_contexto)

print("❓ Pergunta:", pergunta_v2)
print()
print("🤖 Resposta com contexto:")
print("─" * 60)
print(resposta_v2.text)

❓ Pergunta: Qual é a política de férias da Empresa SEMANTIX para funcionários CLT?

🤖 Resposta com contexto:
────────────────────────────────────────────────────────────
A política de férias da Empresa SEMANTIX para funcionários CLT, conforme a Portaria Interna nº 42/2024, é a seguinte:

*   Têm direito a 30 dias corridos de férias por ano.
*   As férias podem ser fracionadas em até 3 períodos, sendo o menor deles não inferior a 14 dias.
*   O período de gozo deve ser comunicado ao RH com 30 dias de antecedência.
*   O pagamento ocorre até 2 dias antes do início das férias.


**Muito melhor!** Agora o modelo responde com precisão, baseado exatamente no documento fornecido.

**Mas há um problema:** coloquei esse contexto **na mão**. No mundo real, com centenas ou milhares de documentos, como faço isso automaticamente?

> Como eu sei **qual trecho** é o relevante para cada pergunta?

Isso nos leva para uma área que existe há **décadas**: **Recuperação de Informação** (*Information Retrieval*).

---

## Parte 2: Recuperação de Informação Clássica

### Breve histórico

| Ano | Marco |
|-----|-------|
| 1945 | Vannevar Bush — *"As We May Think"* / Memex |
| 1960–70 | Gerard Salton — Sistema SMART / **TF-IDF** |
| 1994 | Robertson et al. — **BM25** (ainda usado hoje) |
| 2013 | Mikolov et al. — **Word2Vec** / Embeddings semânticos |
| 2020–21 | Lewis et al. — **RAG** (Facebook AI Research) |

Vamos entender o TF-IDF, um método clássico de recuperação, para entender por que precisamos de algo melhor.

### 📊 Bloco 3 — TF-IDF: Term Frequency × Inverse Document Frequency

**A intuição:**

- **TF (Term Frequency):** quantas vezes o termo aparece neste documento? → mais aparições = mais relevante
- **IDF (Inverse Document Frequency):** em quantos documentos esse termo aparece? → palavra rara = mais informativa; "de", "e", "o" = pouco informativas
- **Score = TF × IDF**

Documentos com palavras raras e específicas ficam no topo do ranking.

In [ ]:
# Bloco 3 — TF-IDF: a forma clássica de recuperação de informação

# Mini corpus: nossos "documentos da empresa"
documentos = [
    "A política de férias permite 30 dias corridos para funcionários CLT.",
    "O reembolso de despesas de viagem deve ser solicitado em até 30 dias.",
    "Funcionários têm direito a plano de saúde após 90 dias de contrato.",
    "As férias podem ser fracionadas em até 3 períodos conforme a portaria.",
    "O relatório financeiro do trimestre será apresentado na reunião de sexta.",
]

# Query do usuário
query_tfidf = "Quais são as regras de férias?"

# Vetorizar com TF-IDF e calcular similaridade
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documentos)
query_vec = vectorizer.transform([query_tfidf])
scores_tfidf = cosine_similarity(query_vec, tfidf_matrix)[0]

print(f"🔍 Query: '{query_tfidf}'\n")
print("📊 Scores de relevância (TF-IDF):\n")
for i, (doc, score) in enumerate(zip(documentos, scores_tfidf)):
    marcador = "✅" if score == max(scores_tfidf) else "  "
    print(f"{marcador} [{score:.3f}] {doc}")

melhor_idx = np.argmax(scores_tfidf)
print(f"\n📌 Documento mais relevante:\n   {documentos[melhor_idx]}")

🔍 Query: 'Quais são as regras de férias?'

📊 Scores de relevância (TF-IDF):

   [0.248] A política de férias permite 30 dias corridos para funcionários CLT.
   [0.148] O reembolso de despesas de viagem deve ser solicitado em até 30 dias.
   [0.148] Funcionários têm direito a plano de saúde após 90 dias de contrato.
✅ [0.401] As férias podem ser fracionadas em até 3 períodos conforme a portaria.
   [0.074] O relatório financeiro do trimestre será apresentado na reunião de sexta.

📌 Documento mais relevante:
   As férias podem ser fracionadas em até 3 períodos conforme a portaria.


Funcionou! O TF-IDF rankeou os documentos sobre férias no topo.

**Agora experimente mudar a forma de perguntar...**

In [ ]:
# Testando o limite do TF-IDF: e se o usuário usar palavras diferentes?

queries_alternativas = [
    "Quais são as regras de férias?",         # query original -> base de comparação
    "Quando posso tirar minhas férias?",       # sinônimo: "tirar" ≠ "regras"
    "Quanto tempo de folga tenho direito?",    # "folga" ≠ "férias"
    "vacation policy for employees",           # em inglês!
]

print("🧪 A mesma pergunta com palavras diferentes:\n")
print("─" * 68)

for query in queries_alternativas:
    q_vec = vectorizer.transform([query])
    scores_q = cosine_similarity(q_vec, tfidf_matrix)[0]
    melhor_idx = np.argmax(scores_q)
    melhor_score = scores_q[melhor_idx]

    print(f"Query  : '{query}'")
    print(f"Score  : {melhor_score:.3f}  →  {documentos[melhor_idx][:55]}...")
    print()

🧪 A mesma pergunta com palavras diferentes:

────────────────────────────────────────────────────────────────────
Query  : 'Quais são as regras de férias?'
Score  : 0.401  →  As férias podem ser fracionadas em até 3 períodos confo...

Query  : 'Quando posso tirar minhas férias?'
Score  : 0.290  →  A política de férias permite 30 dias corridos para func...

Query  : 'Quanto tempo de folga tenho direito?'
Score  : 0.465  →  Funcionários têm direito a plano de saúde após 90 dias ...

Query  : 'vacation policy for employees'
Score  : 0.000  →  A política de férias permite 30 dias corridos para func...



**O problema fica claro:**

- TF-IDF é uma busca **léxica**, ou seja, compara tokens, não significado
- "Tirar férias" e "regras de férias" têm palavras completamente diferentes → score cai
- Em inglês o score chega a zero, mesmo sendo exatamente a mesma pergunta

Precisamos de algo que entenda **significado**, não só palavras.

> Lembram do 1º encontro? Embeddings colocam frases semanticamente similares **próximas no espaço vetorial**. Vamos usar isso.

---

## Parte 3: Busca Semântica com Embeddings

### 🧠 Bloco 4 — Vector Search: busca por significado

**A ideia:**

1. Embeddar todos os documentos → cada um vira uma posição no espaço vetorial
2. Embeddar a query do usuário → posição no mesmo espaço
3. Encontrar os documentos **mais próximos** da query → documentos mais relevantes

A medida de proximidade: **similaridade de cosseno** — o ângulo entre dois vetores.
Independente do comprimento do vetor, o que importa é a **direção**.

In [ ]:
# Bloco 4 — Vector Search: busca semântica com embeddings

# Carregar modelo de embeddings (roda localmente, sem API)
# ⏳ ~30 segundos no primeiro uso (faz download do modelo ~80MB)
print("⏳ Carregando modelo de embeddings...")
modelo_embedding = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Modelo carregado! (384 dimensões por embedding)\n")

⏳ Carregando modelo de embeddings...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Modelo carregado! (384 dimensões por embedding)



In [ ]:
# Embeddar todos os documentos
embeddings_docs = modelo_embedding.encode(documentos)

# Testar as mesmas queries de antes + extras
queries_semanticas = [
    "Quais são as regras de férias?",           # query original
    "Quando posso tirar minhas férias?",         # sinônimo
    "Quanto tempo de folga tenho direito?",      # palavras completamente diferentes
    "vacation policy for employees",              # em inglês!
    "Can I split my vacation into parts?",       # inglês + sinônimo
]

print("🔍 Busca semântica — compare com os resultados do TF-IDF:\n")
print("─" * 68)

for query in queries_semanticas:
    q_emb = modelo_embedding.encode([query])
    scores_sem = cosine_similarity(q_emb, embeddings_docs)[0]
    melhor_idx = np.argmax(scores_sem)

    print(f"Query  : '{query}'")
    print(f"Score  : {scores_sem[melhor_idx]:.3f}  →  {documentos[melhor_idx][:55]}...")
    print()

🔍 Busca semântica — compare com os resultados do TF-IDF:

────────────────────────────────────────────────────────────────────
Query  : 'Quais são as regras de férias?'
Score  : 0.583  →  As férias podem ser fracionadas em até 3 períodos confo...

Query  : 'Quando posso tirar minhas férias?'
Score  : 0.591  →  As férias podem ser fracionadas em até 3 períodos confo...

Query  : 'Quanto tempo de folga tenho direito?'
Score  : 0.420  →  As férias podem ser fracionadas em até 3 períodos confo...

Query  : 'vacation policy for employees'
Score  : 0.574  →  As férias podem ser fracionadas em até 3 períodos confo...

Query  : 'Can I split my vacation into parts?'
Score  : 0.597  →  As férias podem ser fracionadas em até 3 períodos confo...



**Observe:**

- "vacation policy" em inglês → encontrou o documento sobre férias ✅
- "folga" como sinônimo → encontrou corretamente ✅
- Diferentes formas de perguntar → mesma resposta relevante ✅

Isso é o poder dos embeddings: **semântica** sobre forma.

**Novo problema:** se tivermos 50.000 documentos, vamos comparar a query com cada um deles um por um? Precisamos de algo mais eficiente.

E outro problema: o que é exatamente um "documento" para o nosso sistema? Um PDF de 80 páginas pode ser embeddado de uma vez?

Vamos construir o pipeline completo agora.

---

## Parte 4: Construindo o Pipeline RAG

O pipeline RAG tem **dois momentos distintos**:

```
INDEXAÇÃO (feita uma vez, offline):
Documentos → Parsing → Chunks → Embeddings → VectorDB

CONSULTA (a cada pergunta do usuário, online):
Query → Embed → VectorDB → Top-K Chunks → LLM → Resposta
```

Vamos construir cada etapa.

### 📄 Bloco 5 — Parsing: extraindo texto dos documentos

**Parsing** é o processo de extrair conteúdo legível de documentos brutos.

Um PDF pode ser:
- **Texto nativo:** extração direta (PyMuPDF, pdfplumber, pypdf) ← caso simples de hoje
- **Imagem escaneada:** precisa de OCR (Tesseract, Azure Document Intelligence, Google Document AI)
- **Misto:** combinação das duas abordagens

>  Iremos aprofundar em estratégias de parsing nas próximas apresentações.

In [ ]:
# Bloco 5 — Parsing: extraindo texto de PDFs

import fitz  # PyMuPDF

def extrair_texto_pdf(caminho_pdf: str) -> list[dict]:
    """
    Extrai texto de um PDF página por página.

    Args:
        caminho_pdf: caminho para o arquivo .pdf

    Returns:
        Lista de dicts com chaves 'pagina' e 'texto'
    """
    doc = fitz.open(caminho_pdf)
    paginas = []

    for num_pagina, pagina in enumerate(doc):
        texto = pagina.get_text()
        if texto.strip():                         # ignora páginas vazias
            paginas.append({
                "pagina": num_pagina + 1,
                "texto": texto.strip()
            })

    doc.close()
    return paginas


# ── Para o facilitar no Colab, usamos um texto que simula o resultado do parsing ──
texto_simulado = """
POLÍTICA DE RECURSOS HUMANOS — EMPRESA SEMANTIX
Versão 2.0 | Janeiro de 2024

CAPÍTULO 1 — FÉRIAS
Art. 1º - Todo funcionário CLT tem direito a 30 dias corridos de férias
por período aquisitivo de 12 meses.

Art. 2º - As férias podem ser fracionadas em até 3 períodos, sendo o
menor período não inferior a 14 dias corridos.

Art. 3º - O aviso de férias deve ser comunicado ao colaborador com
antecedência mínima de 30 dias.

Art. 4º - O pagamento de férias ocorre até 2 dias antes do início
do período de descanso, acrescido de 1/3 constitucional.

CAPÍTULO 2 — REEMBOLSO DE DESPESAS
Art. 5º - Despesas de viagem a trabalho devem ser solicitadas em até
30 dias após o retorno, mediante comprovante fiscal.

Art. 6º - O reembolso será processado no próximo ciclo de pagamento,
respeitando o calendário financeiro da empresa.

Art. 7º - Despesas sem comprovante fiscal não serão reembolsadas.

CAPÍTULO 3 — PLANO DE SAÚDE
Art. 8º - O plano de saúde corporativo é disponibilizado após 90 dias
de contrato, sem custo para o funcionário.

Art. 9º - O funcionário pode incluir dependentes diretos mediante
solicitação formal ao departamento de RH.
"""

print("✅ Texto extraído com sucesso!")
print(f"📄 Total de caracteres: {len(texto_simulado)}")
print(f"\nPrimeiros 400 caracteres:\n{'─'*40}")
print(texto_simulado[:400])

✅ Texto extraído com sucesso!
📄 Total de caracteres: 1142

Primeiros 400 caracteres:
────────────────────────────────────────

POLÍTICA DE RECURSOS HUMANOS — EMPRESA SEMANTIX
Versão 2.0 | Janeiro de 2024

CAPÍTULO 1 — FÉRIAS
Art. 1º - Todo funcionário CLT tem direito a 30 dias corridos de férias
por período aquisitivo de 12 meses.

Art. 2º - As férias podem ser fracionadas em até 3 períodos, sendo o
menor período não inferior a 14 dias corridos.

Art. 3º - O aviso de férias deve ser comunicado ao colaborador com
antecedê


### ✂️ Bloco 6 — Chunking: dividindo o texto em pedaços

**Por que não embeddar o documento inteiro?**

1. Modelos de embedding têm **limite de tokens**
2. Um embedding de 80 páginas captura o "assunto geral" — perde detalhes específicos
3. Queremos recuperar **o trecho** que responde a pergunta, não o documento inteiro

**Estratégia: Sliding Window com Overlap**

```
Documento: [§1][§2][§3][§4][§5][§6]

Chunk 1:   [§1][§2][§3]
Chunk 2:        [§2][§3][§4]   ← overlap: §2 e §3 aparecem nos dois chunks
Chunk 3:             [§3][§4][§5]
```

O **overlap** evita perder informação importante na fronteira entre chunks.

> Nos próximos encontros vamos aprofundar em estratégias avançadas: chunking semântico, hierárquico, por estrutura de documento.

In [ ]:
# Bloco 6 — Chunking: dividindo o texto em pedaços gerenciáveis

def criar_chunks(texto: str, tamanho_chunk: int = 200, overlap: int = 40) -> list[str]:
    """
    Divide o texto em chunks com janela deslizante (sliding window + overlap).

    Args:
        texto: texto completo a ser dividido
        tamanho_chunk: número aproximado de palavras por chunk
        overlap: número de palavras compartilhadas entre chunks consecutivos

    Returns:
        Lista de strings (chunks)
    """
    palavras = texto.split()
    chunks = []
    inicio = 0

    while inicio < len(palavras):
        fim = min(inicio + tamanho_chunk, len(palavras))
        chunk = " ".join(palavras[inicio:fim])
        chunks.append(chunk)

        if fim == len(palavras):
            break
        inicio += tamanho_chunk - overlap

    return chunks


# Aplicando no texto simulado
chunks = criar_chunks(texto_simulado, tamanho_chunk=80, overlap=15)

print(f"📦 Total de chunks criados: {len(chunks)}\n")
print("─" * 60)

for i, chunk in enumerate(chunks):
    n_palavras = len(chunk.split())
    print(f"Chunk {i+1} ({n_palavras} palavras):")
    print(chunk[:220] + ("..." if len(chunk) > 220 else ""))
    print()

📦 Total de chunks criados: 3

────────────────────────────────────────────────────────────
Chunk 1 (80 palavras):
POLÍTICA DE RECURSOS HUMANOS — EMPRESA SEMANTIX Versão 2.0 | Janeiro de 2024 CAPÍTULO 1 — FÉRIAS Art. 1º - Todo funcionário CLT tem direito a 30 dias corridos de férias por período aquisitivo de 12 meses. Art. 2º - As fé...

Chunk 2 (80 palavras):
férias deve ser comunicado ao colaborador com antecedência mínima de 30 dias. Art. 4º - O pagamento de férias ocorre até 2 dias antes do início do período de descanso, acrescido de 1/3 constitucional. CAPÍTULO 2 — REEMBO...

Chunk 3 (66 palavras):
reembolso será processado no próximo ciclo de pagamento, respeitando o calendário financeiro da empresa. Art. 7º - Despesas sem comprovante fiscal não serão reembolsadas. CAPÍTULO 3 — PLANO DE SAÚDE Art. 8º - O plano de ...



### 🗄️ Bloco 7 — Banco Vetorial: armazenando os embeddings

**Por que não usar um banco de dados relacional?**

| Banco Relacional | Banco Vetorial |
|-----------------|----------------|
| `SELECT * WHERE tema = 'férias'` | `similarity_search(query_emb, k=5)` |
| Busca por valor exato | Retorna os K vetores mais próximos |
| Não entende "proximidade" | Algoritmos ANN: HNSW, FAISS |
| Lento para similaridade em escala | Rápido mesmo com milhões de vetores |

**Ferramentas populares:** ChromaDB · Pinecone · Weaviate · Qdrant · FAISS

Hoje usamos **ChromaDB** — local, sem servidor, instala com pip. Fácil para Colab.

In [ ]:
# Bloco 7 — Vetorização dos chunks + armazenamento no banco vetorial

# Criar banco vetorial em memória
# (ephemeral = dados são perdidos se o runtime reiniciar)
client_db = chromadb.EphemeralClient()

colecao = client_db.create_collection(
    name="documentos_empresa_x",
    metadata={"hnsw:space": "cosine"}              # usa distância cosseno
)

# Gerar embeddings para todos os chunks
print("⚙️  Vetorizando chunks...")
embeddings_chunks = modelo_embedding.encode(chunks).tolist()

# Armazenar no banco vetorial
colecao.add(
    documents=chunks,
    embeddings=embeddings_chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ {len(chunks)} chunks indexados no banco vetorial!")
print(f"\n📊 Exemplo — primeiros 8 valores do embedding do Chunk 0:")
print(f"   {[round(v, 4) for v in embeddings_chunks[0][:8]]} ...")
print(f"   Dimensões totais do embedding: {len(embeddings_chunks[0])}")

⚙️  Vetorizando chunks...
✅ 3 chunks indexados no banco vetorial!

📊 Exemplo — primeiros 8 valores do embedding do Chunk 0:
   [0.0229, 0.1328, 0.0409, -0.0526, 0.1715, -0.0139, -0.128, 0.0278] ...
   Dimensões totais do embedding: 384


### 🔍 Bloco 8 — Retrieval: buscando chunks relevantes

Agora que temos o banco vetorial populado, a cada pergunta do usuário:

1. Gerar o embedding da query
2. Buscar os **K chunks mais similares** no banco
3. Retornar os chunks e suas distâncias

**K é um hiperparâmetro:** mais chunks = mais contexto para o LLM, mas também mais ruído e mais tokens no prompt.

> Ainda vamos explorar estratégias avançadas: busca esparsa (BM25), busca híbrida, re-ranking.

In [ ]:
# Bloco 8 — Retrieval: buscando chunks relevantes para a query

def buscar_chunks_relevantes(query: str, k: int = 3) -> tuple[list[str], list[float]]:
    """
    Busca os k chunks mais relevantes para a query no banco vetorial.

    Args:
        query: pergunta do usuário
        k: número de chunks a retornar

    Returns:
        Tuple (lista de chunks, lista de distâncias)
    """
    query_embedding = modelo_embedding.encode([query]).tolist()

    resultados = colecao.query(
        query_embeddings=query_embedding,
        n_results=k
    )

    chunks_relevantes = resultados["documents"][0]
    distancias = resultados["distances"][0]

    return chunks_relevantes, distancias


# ── Testando o retrieval com diferentes queries ──
queries_retrieval = [
    "Posso parcelar minhas férias?",
    "Como pedir reembolso de uma viagem de trabalho?",
    "Quando começo a ter direito ao plano de saúde?",
]

print("🔍 Testando o Retrieval:\n")
print("─" * 70)

for query in queries_retrieval:
    chunks_encontrados, scores = buscar_chunks_relevantes(query, k=2)
    print(f"❓ Query: '{query}'")
    print(f"📌 Chunk mais relevante (distância cosseno: {scores[0]:.4f}):")
    print(f"   {chunks_encontrados[0][:220]}...")
    print()

🔍 Testando o Retrieval:

──────────────────────────────────────────────────────────────────────
❓ Query: 'Posso parcelar minhas férias?'
📌 Chunk mais relevante (distância cosseno: 0.3927):
   férias deve ser comunicado ao colaborador com antecedência mínima de 30 dias. Art. 4º - O pagamento de férias ocorre até 2 dias antes do início do período de descanso, acrescido de 1/3 constitucional. CAPÍTULO 2 — REEMBO...

❓ Query: 'Como pedir reembolso de uma viagem de trabalho?'
📌 Chunk mais relevante (distância cosseno: 0.3593):
   férias deve ser comunicado ao colaborador com antecedência mínima de 30 dias. Art. 4º - O pagamento de férias ocorre até 2 dias antes do início do período de descanso, acrescido de 1/3 constitucional. CAPÍTULO 2 — REEMBO...

❓ Query: 'Quando começo a ter direito ao plano de saúde?'
📌 Chunk mais relevante (distância cosseno: 0.4163):
   reembolso será processado no próximo ciclo de pagamento, respeitando o calendário financeiro da empresa. Art. 7º - Despesas sem com

### 🚀 Bloco 9 — Pipeline RAG Completo

Agora juntamos tudo:

```
Pergunta → [Embed] → [ChromaDB] → Top-K Chunks ─┐
                                                  ↓
                            Prompt = instrução + contexto + pergunta
                                                  ↓
                                         [Gemini 2.0 Flash]
                                                  ↓
                                             Resposta
```

**As três etapas do RAG:**
- **R**etrieval → buscar chunks relevantes
- **A**ugmented → montar o prompt com o contexto recuperado
- **G**eneration → o LLM gera a resposta baseada no contexto

In [ ]:
# Bloco 9 — Pipeline RAG Completo

def rag_pipeline(pergunta_usuario: str, k: int = 3, verbose: bool = True) -> str:
    """
    Pipeline RAG: Retrieval → Augmentation → Generation.

    Args:
        pergunta_usuario: pergunta em linguagem natural
        k: número de chunks a recuperar
        verbose: se True, mostra detalhes do retrieval

    Returns:
        Resposta gerada pelo LLM (string)
    """
    if verbose:
        print(f"❓ Pergunta: {pergunta_usuario}\n")

    # RETRIEVAL
    chunks_relevantes, scores = buscar_chunks_relevantes(pergunta_usuario, k=k)

    if verbose:
        print(f"📦 {k} chunks recuperados:")
        for chunk, score in zip(chunks_relevantes, scores):
            print(f"   [dist: {score:.4f}] {chunk[:85]}...")
        print()

    # AUGMENTATION
    contexto = "\n\n---\n\n".join(chunks_relevantes)

    prompt = f"""Você é um assistente da Empresa Semantix. Responda a pergunta abaixo usando
APENAS as informações fornecidas no contexto. Se a resposta não estiver
no contexto, diga claramente que não tem essa informação e não invente.

CONTEXTO:
{contexto}

PERGUNTA:
{pergunta_usuario}

RESPOSTA:"""

    # GENERATION
    resposta = model_llm.generate_content(prompt)
    return resposta.text


# Testando o pipeline completo
print("=" * 65)

perguntas_teste = [
    "Posso dividir minhas férias em partes?",
    "Qual o prazo para pedir reembolso de viagem?",
    "Qual é a política de bônus de final de ano?",  # NÃO está nos documentos!
]

for pergunta in perguntas_teste:
    resposta = rag_pipeline(pergunta, k=1)
    print(f"✅ Resposta:\n{resposta}")
    print("=" * 65 + "\n")

❓ Pergunta: Posso dividir minhas férias em partes?

📦 1 chunks recuperados:
   [dist: 0.4301] POLÍTICA DE RECURSOS HUMANOS — EMPRESA SEMANTIX Versão 2.0 | Janeiro de 2024 CAPÍTULO...

✅ Resposta:
Sim, as férias podem ser fracionadas em até 3 períodos, sendo o menor período não inferior a 14 dias corridos.

❓ Pergunta: Qual o prazo para pedir reembolso de viagem?

📦 1 chunks recuperados:
   [dist: 0.2431] férias deve ser comunicado ao colaborador com antecedência mínima de 30 dias. Art. 4º...

✅ Resposta:
O prazo para pedir reembolso de despesas de viagem a trabalho é em até 30 dias após o retorno.

❓ Pergunta: Qual é a política de bônus de final de ano?

📦 1 chunks recuperados:
   [dist: 0.5814] reembolso será processado no próximo ciclo de pagamento, respeitando o calendário fin...

✅ Resposta:
Não tenho essa informação no contexto fornecido.



---

## Conclusão

**RAG não é mágica.** É uma combinação de duas áreas com décadas de história:

- **Recuperação de Informação** → encontrar o trecho certo no momento certo
- **Geração de Linguagem** → formular uma resposta coerente a partir desse trecho

O notebook que você acabou de executar é um **sistema RAG funcional**. Ele é simples, mas de propósito. As próximas apresentações vão mostrar como cada peça pode ser muito mais sofisticada.

---

### O que vem a seguir

| Etapa do Pipeline | Próximos apresentadores |
|-------------------|------------------------|
| **Parsing** | João Henrique · Kalil · Gustavo |
| **Chunking** | Monique · Matheus G. · Gabriel |
| **Retrieval** | Matheus Nery · João Vitor · Davi |